# 02 · Experimento 1 — ablação da cabeça (ResNet-152 fixa)

**Sessão P1 · ambiente L4.** Mesma receita para todos (ResNet-152 ImageNet → Dropout 0,5 → cabeça; 2 fases, 15 + 45 épocas; mesmas dobras; última época; AUC por probabilidade). Só a cabeça muda:

| id | cabeça | parâmetros |
|---|---|---|
| (a) | `Linear(2048,2)` — também é a linha "ResNet152" do Exp. 2 | 4.098 |
| (b) | MLP `[2048,128,2]` (ReLU) | 262.530 |
| (c) | MLP isoparamétrica `[2048,1919,2]` | 3.935.871 |
| (d) | KAN `[2048,128,2]`, grade 10, ordem 3 (referência) | 3.936.000 |
| (e) | KAN `[2048,8,2]` (opcional) | 246.000 |

**Etapas** (`ETAPA`): `"s42"` = (d) → (c) → (b) → (a) com semente 42 (**isto já responde a pergunta**); `"s43_44"` = (d), (c), (b) com sementes 43 e 44 (+ (a) se `INCLUIR_A`); `"e"` = cabeça (e).

**Como rodar:** na 1ª vez deixe `TESTE_RAPIDO = True` (1 dobra × 2 épocas numa pasta de teste); depois mude para `False` e **Executar tudo**. Se a sessão cair, abra de novo e **Executar tudo**: unidades prontas são puladas e a que estava no meio retoma da última época salva. Duas abas ao mesmo tempo: use `ABA = "B"` e `FILTRO_SEMENTES` diferente na segunda (ex.: aba A com 43, aba B com 44).

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o dataset é clonado e o cache é gerado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
ABA = "A"                    # dono das travas desta aba (2ª aba simultânea: "B")
PERMITIR_RESERVA = False     # True só para rodar na T4 como reserva (marcado com † na tabela)
ETAPA = "s42"                # "s42" | "s43_44" | "e"
INCLUIR_A = False            # em "s43_44", inclui também a cabeça (a)
FILTRO_SEMENTES = None       # ex.: [43] numa aba e [44] na outra
TESTE_RAPIDO = True          # 1ª vez True; depois False
CKPT_INTERVALO_MIN = None    # None = usa a recomendação do 01 (0 = checkpoint a cada época)

In [ ]:
# Dependências fixadas (rode ANTES de importar qualquer coisa da albumentations)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "…")
GPU = pc.exigir_gpu(permitir_reserva=PERMITIR_RESERVA)   # rodadas comparáveis só na L4
pc.definir_id_sessao(ABA)
print("GPU:", GPU["nome"], "| comparável:", GPU["comparavel"], "| dono das travas:", pc.ID_SESSAO)

In [ ]:
# Clona o dataset e o leguwoi nos commits fixados e gera o cache 224×224 só nesta sessão (~3–5 min)
sess = pc.preparar_sessao(
    RAIZ_TRABALHO,
    caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}",
    caminho_meta=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_META_DOBRAS}",
)
R, C = f"{DIR_DRIVE}/resultados", f"{DIR_DRIVE}/checkpoints"
RT, CT = f"{DIR_DRIVE}/resultados_teste", f"{DIR_DRIVE}/checkpoints_teste"
print(json.dumps(sess.meta, indent=1))
if sess.meta.get("cache_confere") is False:
    print("⚠️ O cache diverge do registrado no 00 (versão do OpenCV?). Divergência:", sess.meta.get("cache_divergencia"))

In [ ]:
cal = json.loads((Path(R) / "calibracao_l4.json").read_text())
PREC = cal.get("decisao_precisao")
assert PREC in ("amp", "fp32"), "Rode o 01 (ponte fp32 × AMP) antes."
CKPT = cal.get("recomendacao_ckpt_intervalo_min", 0.0) if CKPT_INTERVALO_MIN is None else CKPT_INTERVALO_MIN
E = pc.CONFIGS_EXP1
if ETAPA == "s42":
    configs, sementes = [E["d"], E["c"], E["b"], E["a"]], [42]
elif ETAPA == "s43_44":
    configs, sementes = [E["d"], E["c"], E["b"]] + ([E["a"]] if INCLUIR_A else []), [43, 44]
elif ETAPA == "e":
    configs, sementes = [E["e"]], [42]
else:
    raise ValueError(ETAPA)
if FILTRO_SEMENTES:
    sementes = [s for s in sementes if s in FILTRO_SEMENTES]
fila = pc.montar_fila("ablacao_cabeca", configs, sementes=sementes)
print(f"Precisão: {PREC} | checkpoint a cada ≥ {CKPT} min | {len(fila)} unidades:")
print(pd.DataFrame(fila)[["config", "semente", "dobra"]].groupby(["config", "semente"]).size())

## Teste rápido (1 dobra × 2 épocas, pasta de teste)

In [ ]:
if TESTE_RAPIDO:
    u = {**fila[0], "experimento": "t02_rapido"}
    import shutil
    shutil.rmtree(f"{RT}/t02_rapido", ignore_errors=True); shutil.rmtree(f"{CT}/t02_rapido", ignore_errors=True)
    r = pc.executar_unidade(sess, u, RT, CT, precisao=PREC, limite_epocas=(1, 1), espera_trava_s=0)
    print("Teste rápido OK:", {k: round(v, 4) for k, v in r["metricas"]["es"].items() if k in ("acc", "auc", "f1_macro")},
          "| tempos:", r["tempos"])
    print("➡️ Agora mude TESTE_RAPIDO = False e rode tudo de novo.")

## Rodada

In [ ]:
assert not TESTE_RAPIDO, "TESTE_RAPIDO está True: mude para False para rodar a etapa."
feitos = pc.executar_fila(sess, fila, R, C, precisao=PREC, ckpt_intervalo_min=CKPT)
print("Concluídas nesta sessão:", len(feitos))

## Resultados do experimento (runs.csv, resumo.json, oof.csv)

In [ ]:
def agregar_experimento(experimento):
    """Grava runs.csv (1 linha = modelo × semente × dobra), resumo.json e oof.csv do experimento."""
    runs = pc.ler_resultados(R, experimento)
    if runs.empty:
        print("Nada concluído ainda em", experimento)
        return runs
    runs = runs[~runs["teste_rapido"].astype(bool)].copy()
    d = Path(R) / experimento
    runs.to_csv(d / "runs.csv", index=False)
    metr = ["es_acc", "es_prec_macro", "es_rec_macro", "es_f1_macro", "es_auc"]
    resumo = {}
    for (cfg, sem), g in runs.groupby(["config", "semente"]):
        resumo.setdefault(cfg, {})[f"s{sem}"] = {
            "n_dobras": int(len(g)),
            **{m: {"media": float(g[m].mean()), "dp": float(g[m].std(ddof=1)) if len(g) > 1 else None} for m in metr},
            "treino_min_por_dobra": float(g["treino_min"].mean()),
            "params_cabeca": int(g["params_cabeca"].iloc[0]),
            "params_total": int(g["params_total"].iloc[0]),
            "gpus": sorted(g["gpu"].dropna().unique().tolist()),
        }
    pc.gravar_json_atomico(resumo, d / "resumo.json")
    partes = []
    for _, r in runs.iterrows():
        p = pd.read_csv(Path(r["caminho"]) / "pred_teste.csv")
        partes.append(p.assign(config=r["config"], semente=r["semente"], dobra=r["dobra"]))
    pd.concat(partes).to_csv(d / "oof.csv", index=False)
    print(f"{experimento}: {len(runs)} unidades -> runs.csv, resumo.json, oof.csv")
    return runs

runs = agregar_experimento("ablacao_cabeca")
if not runs.empty:
    display(runs.groupby(["config", "semente"])[["es_acc", "es_f1_macro", "es_auc", "treino_min"]].agg(["mean", "std", "count"]).round(4))

⚠️ Ao terminar, desconecte o ambiente (*Ambiente de execução → Desconectar e excluir*) para não consumir unidades parado.